In [1]:
import pandas as pd
import numpy as np
import yaml
from collections import defaultdict

## Load data

In [2]:
path_to_file = "data/mmc.yaml"

with open(path_to_file, "r", encoding="utf-8") as file:
    raw_data = yaml.safe_load(file)
    
df = pd.DataFrame(raw_data)

df_naselja = pd.read_csv("./processed_data/naselja.csv")

df_regije = df_naselja[['region_id', 'region_name']].drop_duplicates().values

In [3]:
df = df.sample(frac=0.1) #Keep only 10% of all data for testing

In [4]:
df = df[df['paragraphs'].str.len() > 0]
empty_values = ["", "[]"]
df = df[~df['topics'].isin(empty_values)]

In [5]:
# Clean text column, kjer je vse lower case in nima stopwordov

with open('./data/stopwords-sl.txt', 'r', encoding='utf-8') as f:
    sl_stopwords = [line.strip() for line in f if line.strip()]

def clean_data(text_lst):
    full_text = " ".join(text_lst)
    clean_text = "".join(c.lower() if c.isalpha() or c in "čšžćđ-" else " " for c in full_text)
    padded_text = f" {clean_text} "    
    words = padded_text.split()
    final_words = [w for w in words if w not in sl_stopwords]
    return " ".join(final_words)

df["clean_text"] = df["paragraphs"].apply(clean_data)
df["paragraphs"] = df["paragraphs"].apply(
    lambda ps: "\n\n".join(ps) if isinstance(ps, list) else str(ps)
)

In [6]:
# Odstrani naselja, ki imajo več kot 1 lokacijo 

region_counts = df_naselja.groupby('naselje')['region_name'].transform('nunique')
ambiguous_mask = region_counts > 1
df_removed = df_naselja[ambiguous_mask].sort_values(by='naselje')
df_naselja = df_naselja[~ambiguous_mask].copy()
df_naselja = df_naselja.drop_duplicates(subset=['naselje', 'region_name'])

In [7]:
df.head(3)

,_id,url,authors,date,title,paragraphs,figures,lead,mention,topics,keywords,gpt_keywords,id,n_comments,category,clean_text
26060,661f113a1896eefb4be7e644,https://www.rtvslo.si/sport/zimski-sporti/svet...,"[Sara Benkovič, TV Slovenija]",2024-04-16T19:46:20,"Zupančič zapušča skakalke, novi trener naj bi ...",Zupančič je o slovesu razmišljal že pred dvema...,"[{'caption': 'Zoran Zupančič z Niko Prevc, ki ...","Zoran Zupančič, ki se je podpisal pod vse najv...","[Nike Križnar, Nike Prevc]",sport,"[Zoran Zupančič, Jurij Tepeš, smučarski skoki]","[žensko smučarsko skakanje, trener, reprezenta...",705256,92.0,NaN,zupančič slovesu razmišljal dvema letoma premi...
26489,6627a3006ad20d10031915f0,https://www.rtvslo.si/kultura/knjige/znana-des...,[M. K.],2024-04-23T09:56:28,Znana deseterica v boju za kresnika. Med nomin...,Prvi izbor strokovne žirije za Delovo nagrado ...,"[{'caption': 'Na kresno noč, 23. junija, bo zn...",Na svetovni dan knjige smo dobili prvi izbor n...,"[Laure Buzeti, Borisa Kolarja, Blaža Kutina, E...",kultura,"[kresnik, nominiranci, Delo]","[knjiga, nagrada, kresnik, romani, izbor, nomi...",705966,9.0,NaN,izbor strokovne žirije delovo nagrado kresnik ...
22336,6606d051117a5c5285edb708,https://www.rtvslo.si/sport/zimski-sporti/svet...,[T. O.],2024-01-14T10:44:11,Avstrijski praznik tokrat s svojo 40. zmago pr...,Če je v soboto na smuku dvojno avstrijsko zmag...,[{'caption': '32-letna Lara Gut Behrami je zma...,Tridnevni program v Zauchenseeju so alpske smu...,[Manjka samozaupanje],sport,"[alpsko smučanje, superveleslalom, Ilka Štuhec...","[smučanje, superveleslalom, tekma, Ilka Štuhec...",694747,28.0,NaN,soboto smuku dvojno avstrijsko zmagoslavje pre...


## Find regions for each data

In [38]:
#! FOUND EVEN FASTER WAY

# import re
# from collections import defaultdict

# # ─────────────────────────────────────────────────────────────────────────────
# # 1. Pre‑build mapping: term → set of region_id (once, before the loop)
# # ─────────────────────────────────────────────────────────────────────────────
# term_to_region = defaultdict(set)

# for _, row in df_naselja.iterrows():
#     rid = row['region_id']
#     for col in ('naselje', 'rodilnik', 'mestnik'):
#         term = str(row[col])
#         if term and term.lower() != 'nan':
#             term_to_region[term].add(rid)   # keep original case

# # ─────────────────────────────────────────────────────────────────────────────
# # 2. Build regex pattern from all terms (same as before)
# # ─────────────────────────────────────────────────────────────────────────────
# all_terms = set(term_to_region.keys())
# pattern = '|'.join(re.escape(term) for term in all_terms)
# regex = re.compile(rf'\b({pattern})\b', flags=re.IGNORECASE)

# # ─────────────────────────────────────────────────────────────────────────────
# # 3. Fast lookup function – no DataFrame scan per row
# # ─────────────────────────────────────────────────────────────────────────────
# def get_intersected_regions_fast(text_list):
#     full_text = " ".join(text_list)
#     found_words = set(regex.findall(full_text))

#     if not found_words:
#         return None

#     # Collect region IDs directly from the pre‑built mapping
#     region_ids = set()
#     for word in found_words:
#         # Keep original case for lookup (map is case‑sensitive)
#         # But regex.findall with IGNORECASE returns the word *as found*.
#         # If your terms have different case variants, unify them:
#         # word_lower = word.lower()
#         # for key in term_to_region: if key.lower() == word_lower ...
#         # Simpler: store mapping with lower‑case keys and lower‑case the found word.
#         #
#         # The clearest approach (avoids case mismatch):
#         # Build term_to_region with lower‑case keys, then use word.lower().
#         region_ids.update(term_to_region.get(word, set()))
#     return list(region_ids)

# # 4. Apply
# df['intersected_regions'] = df['paragraphs'].apply(get_intersected_regions_fast)

### Brez naselji

In [ ]:
# term_to_region = defaultdict(set)

# for _, row in df_naselja.iterrows():
#     rid = row['region_id']
#     for col in ('naselje', 'rodilnik', 'mestnik'):
#         term = str(row[col])
#         if term and term.lower() != 'nan':
#             term_to_region[term].add(rid)

In [ ]:
# import ahocorasick

# # Build automaton with lowercase terms
# automaton = ahocorasick.Automaton()
# for term, region_ids in term_to_region.items():
#     spaced_term1 = f" {term.lower()} " # Beseda more imeti space okrog sebe
#     automaton.add_word(spaced_term1, (list(region_ids), term))  # Store just the IDs
# automaton.make_automaton()

# def get_regions_ahocorasick_fixed(text):
#     region_ids = set()
#     found_matches = {} 
    
#     for _, (ids, word) in automaton.iter(text):
#         region_ids.update(ids)
#         found_matches[word] = ids

#     #if found_matches:
#         # Create a list of "Word (ID, ID)" strings
#     #    display_list = [f"{word} {ids}" for word, ids in found_matches.items()]
#     #    print(f"Matched: {', '.join(display_list)}")
    
#     return list(region_ids) if region_ids else None
# df['intersected_regions'] = df['clean_text'].apply(get_regions_ahocorasick_fixed)


### Get also the naselja

In [36]:
from collections import defaultdict

term_to_data = {}

for _, row in df_naselja.iterrows():
    rid = row['region_id']
    base_naselje = str(row['naselje'])
    
    for col in ('naselje', 'rodilnik', 'mestnik'):
        term = str(row[col])
        if term and term.lower() != 'nan':
            term_lower = term.lower()
            
            # Store the tuple directly as the target payload
            if term_lower not in term_to_data:
                term_to_data[term_lower] = (rid, base_naselje)

In [37]:
import ahocorasick

automaton = ahocorasick.Automaton()

for term_lower, location_tuple in term_to_data.items():
    spaced_term = f" {term_lower} " 
    automaton.add_word(spaced_term, location_tuple)

automaton.make_automaton()

In [38]:
def get_regions_and_naselja_tuples(text):
    region_ids = set()
    matched_tuples = set() # Using a set automatically handles duplicate word hits in text
    
    # payload is now the tuple: (region_id, base_naselje)
    for _, (rid, base_naselje) in automaton.iter(text):
        region_ids.add(rid)
        matched_tuples.add((rid, base_naselje))

    if not region_ids:
        return None, None
        
    return list(region_ids), list(matched_tuples)

# 4. Apply to your dataframe
res = df['clean_text'].apply(get_regions_and_naselja_tuples).dropna()

df['intersected_regions'] = res.apply(lambda x: x[0] if x else None)
df['intersected_naselja'] = res.apply(lambda x: x[1] if x else None)

# Drop rows that didn't match any region/naselje
novice_z_naselji_df = df[df['intersected_regions'].notna()].copy()

In [41]:
novice_z_naselji_df

,_id,url,authors,date,title,paragraphs,figures,lead,mention,topics,keywords,gpt_keywords,id,n_comments,category,clean_text,intersected_regions,intersected_naselja
26489,6627a3006ad20d10031915f0,https://www.rtvslo.si/kultura/knjige/znana-des...,[M. K.],2024-04-23T09:56:28,Znana deseterica v boju za kresnika. Med nomin...,Prvi izbor strokovne žirije za Delovo nagrado ...,"[{'caption': 'Na kresno noč, 23. junija, bo zn...",Na svetovni dan knjige smo dobili prvi izbor n...,"[Laure Buzeti, Borisa Kolarja, Blaža Kutina, E...",kultura,"[kresnik, nominiranci, Delo]","[knjiga, nagrada, kresnik, romani, izbor, nomi...",705966,9.0,NaN,izbor strokovne žirije delovo nagrado kresnik ...,"[SI034, SI041]","[(SI034, Vine), (SI041, Rožnik)]"
22336,6606d051117a5c5285edb708,https://www.rtvslo.si/sport/zimski-sporti/svet...,[T. O.],2024-01-14T10:44:11,Avstrijski praznik tokrat s svojo 40. zmago pr...,Če je v soboto na smuku dvojno avstrijsko zmag...,[{'caption': '32-letna Lara Gut Behrami je zma...,Tridnevni program v Zauchenseeju so alpske smu...,[Manjka samozaupanje],sport,"[alpsko smučanje, superveleslalom, Ilka Štuhec...","[smučanje, superveleslalom, tekma, Ilka Štuhec...",694747,28.0,NaN,soboto smuku dvojno avstrijsko zmagoslavje pre...,"[SI037, SI042]","[(SI037, Ograja), (SI037, Smuka), (SI042, Forme)]"
50764,67f5b8afc06756965411a58d,https://www.rtvslo.si/sport/hokej/alpska-liga/...,[S. J.],2025-04-08T17:57:12,Gol po vsega 12 sekundah tlakoval visoko zmago...,Železarji. Varovancem Marcela Rodmana so zadal...,[{'caption': 'Avstrijci so takoj zadeli in vod...,Hokejisti Zell am Seeja so na drugi tekmi fina...,"[Aljaž Predan, Nick Huard, Erik Svetina, Kilia...",sport,"[alpska liga, Jesenice, finale, Zell am See]",NaN,742133,NaN,NaN,železarji varovancem marcela rodmana zadali sp...,"[SI034, SI043, SI036]","[(SI036, Dunaj), (SI043, Polje), (SI043, Jesen..."
45065,678cf78b3e2f7cf55b188f5a,https://www.rtvslo.si/zabava-in-slog/glasba/em...,[Žana E. Čeh],2025-01-19T14:00:00,"KiKi: Ema se mi zdi priložnost, da si lahko ze...","KiKi je glasbo pred leti začela ustvarjati "" č...","[{'caption': 'KiKi.', 'img': 'https://img.rtvc...","""Zdi se mi, da je moja glasba, ki je morda obč...","[Roku Molnarju, Laetitia Pohl,, Bernarda Žarn,...",zabava-in-slog,"[KiKi, Ema 2025, Ema pred Emo, Evrovizija, O-ou!]",NaN,733412,4.0,NaN,kiki glasbo leti začela ustvarjati čisto amate...,[SI041],"[(SI041, Ljubljana)]"
35884,66db97bb7cff1ce3387558be,https://www.rtvslo.si/slovenija/tomaz-vesel-od...,[La. Da.],2024-09-06T14:52:05,Tomaž Vesel odstopil od kandidature za evropsk...,"Iz kabineta predsednika vlade so sporočili, da...",[{'caption': 'Tomaž Vesel je svojo kandidaturo...,Premier Robert Golob je sprejel odstop Tomaža ...,"[La. Da., Robert Golob, Tomaža Vesela, Janez J...",slovenija,"[Slovenski kandidat, Evropska komisija, Ursula...",NaN,720232,561.0,NaN,kabineta predsednika vlade sporočili premier r...,[SI034],"[(SI034, Imeno)]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51783,68097f3bef143c5d4ba9972d,https://www.rtvslo.si/slovenija/poslanke-svobo...,[T. L. Š.],2025-04-23T18:46:49,Poslanke Svobode vložile tožbo zaradi upodobit...,"Po navedbah N1 so v tožbi, ki so jo vložile na...","[{'caption': 'Sporna naslovnica Demokracije.',...",Poslanke Gibanja Svoboda Urška Klakočar Zupanč...,"[Tamara Vonta, Lena Grgurevič]",slovenija,"[Nova obzorja, Klakočar Zupančič, Demokracija,...",NaN,743674,NaN,NaN,navedbah tožbi vložile ljubljansko okrožno sod...,[SI036],"[(SI036, Raztez)]"
40787,6739404cfa17aa449d92c1c4,https://www.rtvslo.si/slovenija/na-presernovem...,[T. L. Š.],2024-11-16T12:15:10,"Na Prešernovem trgu protivojni shod za mir. ""D...","""Pobuda za današnji shod je nastala spontano, ...",[{'caption': 'Shod na Prešernovem trgu v Ljubl...,Na ljubljanskem Prešernovem trgu je potekal mi...,"[Anica Kos, Uroš Lipušček, Nataša Posel, Rudi ...",slovenija,"[Ljubljana, pohod za mir, Ne v mojem imenu!]",NaN,727652,84.0,NaN,pobuda današnji shod nastala sp

In [40]:
novice_z_naselji_df = df[df['intersected_regions'].astype(bool)]

In [ ]:
novice_z_naselji_df.count()

_id                    3572
url                    3572
authors                3501
date                   3572
title                  3572
paragraphs             3572
figures                3572
lead                   3571
mention                3572
topics                 3572
keywords               3572
gpt_keywords           1387
id                     3572
n_comments             2351
category                 11
clean_text             3572
intersected_regions    3572
intersected_naselja    3572
dtype: int64

In [43]:
novice_z_naselji_df_za_db = novice_z_naselji_df[["id", "title", "url", "date", "topics", "paragraphs", "intersected_regions", "clean_text", "intersected_naselja"]]

## Vectorize with TFIDF paragraphs

In [44]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vec = TfidfVectorizer(max_features=500, max_df=0.3, min_df=3) # Vektorji velikosti 500, max_df=0.3 => če se beseda pojavi 30%+ časa jo ignoriraj, min_df = minimalno kolikokrat se rabi beseda pojavit
tfidf_matrix = tfidf_vec.fit_transform(novice_z_naselji_df_za_db['clean_text'].fillna(''))
novice_z_naselji_df_za_db['tfidf'] = list(tfidf_matrix.toarray())

In [45]:
feature_names = tfidf_vec.get_feature_names_out()
np.save('final_data/tfidf_vocab.npy', feature_names, allow_pickle=True)

In [52]:
novice_z_naselji_df_za_db.columns

Index(['id', 'title', 'url', 'date', 'topics', 'paragraphs',
       'intersected_regions', 'clean_text', 'intersected_naselja', 'tfidf'],
      dtype='str')

# Save to db

In [53]:
import sqlite3
print(sqlite3.sqlite_version)

3.45.3


In [54]:
connection = sqlite3.connect('final_data/novice.db') # Naredi db če še ne obstaja
connection.execute("PRAGMA foreign_keys = ON")
cursor = connection.cursor()

In [ ]:
# Naredi db
#cursor.execute("DELETE FROM novice")
#cursor.execute("DELETE FROM regije")

cursor.execute('''
    CREATE TABLE IF NOT EXISTS regije (
        id CHAR(5) PRIMARY KEY,
        name VARCHAR(50)
    )
''')
cursor.executemany("INSERT OR IGNORE INTO regije (id, name) VALUES (?, ?)", df_regije)

cursor.execute('''
    CREATE TABLE IF NOT EXISTS novice (
        id INTEGER PRIMARY KEY,
        title TEXT,
        url TEXT,
        date DATE,
        topic VARCHAR(30),
        content TEXT,
        clean_content TEXT,
        tfidf BLOB
    )
''')

# 2. Junction table for Regions
cursor.execute('''
    CREATE TABLE IF NOT EXISTS novice_regije (
        novica_id INTEGER,
        regija_id CHAR(5),
        PRIMARY KEY (novica_id, regija_id),
        FOREIGN KEY (novica_id) REFERENCES novice (id) ON DELETE CASCADE,
        FOREIGN KEY (regija_id) REFERENCES regije (id) ON DELETE CASCADE
    )
''')

# 3. Junction table for Naselja (includes regija_id to preserve the tuple)
cursor.execute('''
    CREATE TABLE IF NOT EXISTS novice_naselja (
        novica_id INTEGER,
        regija_id CHAR(5),
        naselje VARCHAR(100),
        PRIMARY KEY (novica_id, regija_id, naselje),
        FOREIGN KEY (novica_id) REFERENCES novice (id) ON DELETE CASCADE,
        FOREIGN KEY (regija_id) REFERENCES regije (id) ON DELETE CASCADE
    )
''')

cursor.execute("CREATE INDEX IF NOT EXISTS idx_novice_topic ON novice (topic)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_novice_date ON novice (date)")
connection.commit()

In [ ]:
import pickle
import sqlite3
import json

def save_data_to_db(connection, df):
    cursor = connection.cursor()
    
    novice_items = []
    regije_junction_items = []
    naselja_junction_items = []
    
    for _, row in df.iterrows():
        n_id = row['id']
        
        # 1. Convert TFIDF numpy array into a SQLite-compatible BLOB binary
        tfidf_blob = sqlite3.Binary(pickle.dumps(row['tfidf'])) if row['tfidf'] is not None else None
        
        # 2. Safely stringify the topics column (handling lists/strings)
        topics_val = row['topics']
        if isinstance(topics_val, list):
            # Option A: Convert to comma-separated string e.g., "Šport, Kronika"
            # topics_str = ", ".join(str(t) for t in topics_val)
            # Option B: Store as valid JSON string so you can read it back as a list later
            topics_str = json.dumps(topics_val, ensure_ascii=False)
        else:
            topics_str = str(topics_val) if pd.notna(topics_val) else ""

        # Append data aligned precisely with the 'novice' table schema
        novice_items.append((
            n_id,
            row['title'],
            row['url'],
            row['date'],
            topics_str,
            row['paragraphs'],
            row['clean_text'],
            tfidf_blob
        ))
        
        # 3. Explode the intersected_regions list
        r_ids = row['intersected_regions']
        if isinstance(r_ids, list):
            for r_id in r_ids:
                regije_junction_items.append((n_id, r_id))
                
        # 4. Explode the intersected_naselja tuples list [(region_id, naselje), ...]
        nas_tuples = row['intersected_naselja']
        if isinstance(nas_tuples, list):
            for r_id, naselje in nas_tuples:
                naselja_junction_items.append((n_id, r_id, naselje))

    try:
        # Insert into main news table
        cursor.executemany('''
            INSERT OR REPLACE INTO novice (id, title, url, date, topic, content, clean_content, tfidf) 
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ''', novice_items)
        
        # Insert into novice_regije junction table
        cursor.executemany('''
            INSERT OR IGNORE INTO novice_regije (novica_id, regija_id) 
            VALUES (?, ?)
        ''', regije_junction_items)
        
        # Insert into novice_naselja junction table
        cursor.executemany('''
            INSERT OR IGNORE INTO novice_naselja (novica_id, regija_id, naselje) 
            VALUES (?, ?, ?)
        ''', naselja_junction_items)
        
        connection.commit()
        print(f"Successfully saved {len(df)} articles.")
        
    except Exception as e:
        connection.rollback()
        print(f"Error during save: {e}")
    finally:
        cursor.close()

#! UNCOMMENT IF NEED
save_data_to_db(connection, novice_z_naselji_df_za_db)

Successfully saved 3572 articles.


In [58]:
connection.close()